In [ ]:
# -------------------------- ROUND 1 ------------------------------------
# Exercise 1
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7],
    "age": [25, 42, np.nan, 35, 29, 51, 42],
    "income": [30000, 55000, 42000, np.nan, 38000, 90000, 55000],
    "contract": [
        "monthly",
        "yearly",
        "monthly",
        "monthly",
        "yearly",
        "yearly",
        "monthly"
    ],
    "churn": [1, 0, 1, 0, 0, 0, 1]
})

df["age"] = df["age"].fillna(df["age"].median())
df["income"] = df["income"].fillna(df["income"].median())
df["contract"] = df["contract"].apply(lambda x: 1 if x == "monthly" else 0)
df["income_per_age"] = df["income"] / df["age"]
print(df)

# Exercise 2
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3, 3, 3, 4],
    "date": [
        "2026-01-10",
        "2026-02-15",
        "2026-03-20",
        "2026-02-01",
        "2026-03-05",
        "2026-01-15",
        "2026-01-20",
        "2026-03-01",
        "2026-02-10"
    ],
    "amount": [100, 50, 200, 300, 100, 20, 30, 80, 500]
})

reference_date = pd.Timestamp("2026-04-01")

grouped_df = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    average_transaction=("amount", "mean"),
    number_of_transactions=("amount", "count"),
    last_transaction=("date", "max"),
)

grouped_df["days_since_last_transaction"] = reference_date - pd.to_datetime(grouped_df["last_transaction"])

print(grouped_df)


# Exercise 3
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "country": ["ES", "ES", "FR", "FR"],
    "segment": ["A", "B", "A", "B"]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3, 4, 4],
    "amount": [100, 50, 200, 100, 50, 300, 20, 80]
})


transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
)

customer_data = customers.copy()

customer_data = customer_data.merge(transactions, on="customer_id", how="left")

print(customer_data)

country_segment_data = customer_data.groupby(["country", "segment"]).agg(
    average_customer_spend=("total_spend", "mean"),
    number_customers=("total_spend", "count"),
)

print(country_segment_data)


# Exercise 4
df = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3],
    "purchase": [100, 200, 50, 100, 150, 500]
})

df["customer_total"] = df.groupby("customer_id")["purchase"].transform("sum")

df["purchase_share"] = round(df["purchase"] / df["customer_total"] * 100, 2)

print(df)

# Exercise 5
df = pd.DataFrame({
    "country": ["ES", "ES", "ES", "FR", "FR", "FR"],
    "customer": ["A", "B", "C", "D", "E", "F"],
    "revenue": [1000, 500, 1000, 200, 500, 300]
})

df["revenue_rank"] = df.groupby("country")["revenue"].rank(method="dense", ascending=False).astype(int)

print(df)

# -------------------------- ROUND 2 ------------------------------------
# Exercise 6
df = pd.DataFrame({
    "age": [25, 35, 42, 51, 29],
    "income": [30000, 50000, np.nan, 90000, 40000],
    "city": ["Madrid", "Madrid", "Barcelona", "Madrid", "Barcelona"],
    "contract": ["monthly", "yearly", "monthly", "yearly", "monthly"],
    "churn": [1, 0, 1, 0, 1]
})

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


y = df["churn"]
X = df.drop(columns=["churn"])

print(X.dtypes)

categorical_columns = [col for col in X.columns if X[col].dtype == "str"]
numerical_columns = [col for col in X.columns if col not in categorical_columns]

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical_imputing", SimpleImputer(strategy="median"), numerical_columns),
        ("numerical_encoding", StandardScaler(), numerical_columns),
        ("categorical_imputing", SimpleImputer(strategy="most_frequent"), numerical_columns),
        ("categorical_encoding", OneHotEncoder(), numerical_columns),
    ]
)

print(df)

preprocessed_df = preprocessor.fit_transform(X, y)

print(preprocessed_df)

# Exercise 7
df = pd.DataFrame({
    "customer_id": range(1, 11),
    "monthly_spend": [
        20, 25, 30, 35, 40,
        42, 45, 50, 55, 5000
    ]
})

print(df)

Q3 = df["monthly_spend"].quantile(0.75)
Q1 = df["monthly_spend"].quantile(0.25)

IQR = Q3 - Q1
median = df["monthly_spend"].median()

admissible_distance = 1.5 * IQR

for i, row_df in df.iterrows():
    if abs(row_df["monthly_spend"] - median) > admissible_distance:
        print(f"Potential outlier: Row with customer id {row_df["customer_id"]} has monthly spend {row_df["monthly_spend"]}, outside the admissible range")


# Exercise 8
ages = np.array([15, 22, 35, 67, 42, 12, 29])

age_group = np.where(ages < 18, "minor", np.where(ages > 64, "senior", "adult"))

print(ages)
print(age_group)

# Exercise 9
df = pd.DataFrame({
    "monthly_spend": [20, 100, 250, 80, 500],
    "number_products": [1, 2, 5, 1, 8]
})

df["customer_value"] = df.apply(
    lambda x: "high" if (x["monthly_spend"] >= 200 and x["number_products"] >= 5) else ("medium" if (x["monthly_spend"] >= 100 or x["number_products"] >= 3) else "low"),
    axis=1
)

print(df)


# Exercise 10
df = pd.DataFrame({
    "customer_id": [1, 2, 2, 3, 4, 4],
    "date": [
        "2026-01-01",
        "2026-01-03",
        "2026-02-10",
        "2026-01-05",
        "2026-01-01",
        "2026-03-01"
    ],
    "income": [30000, 50000, 55000, 40000, 70000, 72000]
})

df = df.sort_values(by=["customer_id", "date"], ascending=[True, False]).drop_duplicates("customer_id", keep="first")

print(df)


# -------------------------- ROUND 3 ------------------------------------

# Exercise 11
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "country": ["ES", "ES", "FR", "FR", "ES"],
    "age": [25, 40, 35, np.nan, 50]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 3, 3, 5],
    "date": [
        "2026-01-10",
        "2026-02-10",
        "2026-01-15",
        "2026-03-10",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-02-20"
    ],
    "amount": [100, 200, 50, 100, 300, 100, 200, 500]
})

reference_date = pd.Timestamp("2026-04-01")

transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
    last_date=("date", "max"),
    # days_since_last_transaction=("amount", "mean"),
)

transactions["days_since_last_transaction"] = (reference_date - pd.to_datetime(transactions["last_date"])).dt.days
transactions = transactions.drop(columns=["last_date"])

customer_data = customers.merge(transactions, on="customer_id", how="left")

# customer_data["age"] = customer_data["age"].fillna(customer_data["age"].median())

print(customer_data)

# Exercise 12
df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "age": [25, 40, 35, 50, 29],
    "monthly_spend": [50, 100, 80, 200, 40],
    "support_calls_next_30d": [2, 5, 0, 8, 1],
    "churn_next_30d": [0, 1, 0, 1, 0]
})

# Knowing that churn_next_30_d is the target we want to predict. We leave that as the target. Related to the features, we
# can not use support_call_next_30d because their values are not present at prediction time so that would be data leakage.
# So, independently of their good predictive power, it can not be used for that reason.


# Exercise 13
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3],
    "date": pd.to_datetime([
        "2026-01-01",
        "2026-02-01",
        "2026-03-15",
        "2026-01-10",
        "2026-03-01",
        "2026-02-15"
    ]),
    "amount": [100, 200, 50, 300, 100, 500]
})

prediction_date = pd.Timestamp("2026-03-01")

customer_spends = transactions[(prediction_date - transactions["date"]).dt.days > 0]

customer_spends = customer_spends.groupby("customer_id").agg(
    total_spend=("amount", "sum")
)

print(customer_spends)

# Exercise 15
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "country": ["ES", "ES", "FR", "FR", "ES"],
    "age": [25, 40, 35, 50, np.nan]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 3, 3, 5, 5],
    "date": pd.to_datetime([
        "2026-01-10",
        "2026-02-10",
        "2026-01-15",
        "2026-03-10",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-01-20",
        "2026-02-20"
    ]),
    "amount": [100, 200, 50, 100, 300, 100, 200, 100, 500]
})

reference_date = pd.Timestamp("2026-04-01")

transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
    last_date=("date", "max"),
)

transactions["days_since_last_transaction"] = (reference_date - transactions["last_date"]).dt.days
transactions.drop(columns=["last_date"])

customer_data = customers.merge(transactions, on="customer_id", how="left")
print(customer_data)

In [ ]:
# Exercise 1
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7],
    "age": [25, 42, np.nan, 35, 29, 51, 42],
    "income": [30000, 55000, 42000, np.nan, 38000, 90000, 55000],
    "contract": [
        "monthly",
        "yearly",
        "monthly",
        "monthly",
        "yearly",
        "yearly",
        "monthly"
    ],
    "churn": [1, 0, 1, 0, 0, 0, 1]
})

df["age"] = df["age"].fillna(df["age"].median())
df["income"] = df["income"].fillna(df["income"].median())
df["contract"] = df["contract"].apply(lambda x: 1 if x == "monthly" else 0)
df["income_per_age"] = df["income"] / df["age"]
print(df)

   customer_id   age   income  contract  churn  income_per_age
0            1  25.0  30000.0         1      1     1200.000000
1            2  42.0  55000.0         0      0     1309.523810
2            3  38.5  42000.0         1      1     1090.909091
3            4  35.0  48500.0         1      0     1385.714286
4            5  29.0  38000.0         0      0     1310.344828
5            6  51.0  90000.0         0      0     1764.705882
6            7  42.0  55000.0         1      1     1309.523810


In [90]:
# Exercise 2
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3, 3, 3, 4],
    "date": [
        "2026-01-10",
        "2026-02-15",
        "2026-03-20",
        "2026-02-01",
        "2026-03-05",
        "2026-01-15",
        "2026-01-20",
        "2026-03-01",
        "2026-02-10"
    ],
    "amount": [100, 50, 200, 300, 100, 20, 30, 80, 500]
})

reference_date = pd.Timestamp("2026-04-01")

grouped_df = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    average_transaction=("amount", "mean"),
    number_of_transactions=("amount", "count"),
    last_transaction=("date", "max"),
)

grouped_df["days_since_last_transaction"] = reference_date - pd.to_datetime(grouped_df["last_transaction"])

print(grouped_df)


             total_spend  average_transaction  number_of_transactions  \
customer_id                                                             
1                    350           116.666667                       3   
2                    400           200.000000                       2   
3                    130            43.333333                       3   
4                    500           500.000000                       1   

            last_transaction days_since_last_transaction  
customer_id                                               
1                 2026-03-20                     12 days  
2                 2026-03-05                     27 days  
3                 2026-03-01                     31 days  
4                 2026-02-10                     50 days  


In [26]:
# Exercise 3
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "country": ["ES", "ES", "FR", "FR"],
    "segment": ["A", "B", "A", "B"]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3, 4, 4],
    "amount": [100, 50, 200, 100, 50, 300, 20, 80]
})


transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
)

customer_data = customers.copy()

customer_data = customer_data.merge(transactions, on="customer_id", how="left")

print(customer_data)

country_segment_data = customer_data.groupby(["country", "segment"]).agg(
    average_customer_spend=("total_spend", "mean"),
    number_customers=("total_spend", "count"),
)

print(country_segment_data)

   customer_id country segment  total_spend  number_transactions  \
0            1      ES       A          150                    2   
1            2      ES       B          350                    3   
2            3      FR       A          300                    1   
3            4      FR       B          100                    2   

   average_transaction  
0            75.000000  
1           116.666667  
2           300.000000  
3            50.000000  
                 average_customer_spend  number_customers
country segment                                          
ES      A                         150.0                 1
        B                         350.0                 1
FR      A                         300.0                 1
        B                         100.0                 1


In [32]:
# Exercise 4
df = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3],
    "purchase": [100, 200, 50, 100, 150, 500]
})

df["customer_total"] = df.groupby("customer_id")["purchase"].transform("sum")

df["purchase_share"] = round(df["purchase"] / df["customer_total"] * 100, 2)

print(df)

   customer_id  purchase  customer_total  purchase_share
0            1       100             300           33.33
1            1       200             300           66.67
2            2        50             300           16.67
3            2       100             300           33.33
4            2       150             300           50.00
5            3       500             500          100.00


In [ ]:
# Exercise 5
df = pd.DataFrame({
    "country": ["ES", "ES", "ES", "FR", "FR", "FR"],
    "customer": ["A", "B", "C", "D", "E", "F"],
    "revenue": [1000, 500, 1000, 200, 500, 300]
})

df["revenue_rank"] = df.groupby("country")["revenue"].rank(method="dense", ascending=False).astype(int)

print(df)

  country customer  revenue  revenue_rank
0      ES        A     1000             1
1      ES        B      500             2
2      ES        C     1000             1
3      FR        D      200             3
4      FR        E      500             1
5      FR        F      300             2


In [92]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

df = pd.DataFrame({
    "age": [25, 35, 42, 51, 29],
    "income": [30000, 50000, np.nan, 90000, 40000],
    "city": ["Madrid", "Madrid", "Barcelona", "Madrid", "Barcelona"],
    "contract": ["monthly", "yearly", "monthly", "yearly", "monthly"],
    "churn": [1, 0, 1, 0, 1]
})

y = df["churn"]
X = df.drop(columns=["churn"])

categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns
numerical_columns = X.select_dtypes(include=["number"]).columns

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

X_preprocessed = preprocessor.fit_transform(X)
print(X_preprocessed)

[[-1.2275819  -1.01985016  0.          1.          1.          0.        ]
 [-0.15075567 -0.04856429  0.          1.          0.          1.        ]
 [ 0.60302269 -0.29138576  1.          0.          1.          0.        ]
 [ 1.5721663   1.89400743  0.          1.          0.          1.        ]
 [-0.79685141 -0.53420722  1.          0.          1.          0.        ]]


In [95]:
# Exercise 7
df = pd.DataFrame({
    "customer_id": range(1, 11),
    "monthly_spend": [
        20, 25, 30, 35, 40,
        42, 45, 50, 55, 5000
    ]
})

Q1 = df["monthly_spend"].quantile(0.25)
Q3 = df["monthly_spend"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["monthly_spend"] < lower_bound) |
    (df["monthly_spend"] > upper_bound)
]

print(outliers)

   customer_id  monthly_spend
9           10           5000


In [58]:
# Exercise 8
ages = np.array([15, 22, 35, 67, 42, 12, 29])

age_group = np.where(ages < 18, "minor", np.where(ages > 64, "senior", "adult"))

print(ages)
print(age_group)

[15 22 35 67 42 12 29]
['minor' 'adult' 'adult' 'senior' 'adult' 'minor' 'adult']


In [61]:
# Exercise 9
df = pd.DataFrame({
    "monthly_spend": [20, 100, 250, 80, 500],
    "number_products": [1, 2, 5, 1, 8]
})

df["customer_value"] = df.apply(
    lambda x: "high" if (x["monthly_spend"] >= 200 and x["number_products"] >= 5) else ("medium" if (x["monthly_spend"] >= 100 or x["number_products"] >= 3) else "low"),
    axis=1
)

print(df)

   monthly_spend  number_products customer_value
0             20                1            low
1            100                2         medium
2            250                5           high
3             80                1            low
4            500                8           high


In [62]:
# Exercise 10
df = pd.DataFrame({
    "customer_id": [1, 2, 2, 3, 4, 4],
    "date": [
        "2026-01-01",
        "2026-01-03",
        "2026-02-10",
        "2026-01-05",
        "2026-01-01",
        "2026-03-01"
    ],
    "income": [30000, 50000, 55000, 40000, 70000, 72000]
})

df = df.sort_values(by=["customer_id", "date"], ascending=[True, False]).drop_duplicates("customer_id", keep="first")

print(df)

   customer_id        date  income
0            1  2026-01-01   30000
2            2  2026-02-10   55000
3            3  2026-01-05   40000
5            4  2026-03-01   72000


In [ ]:
# Exercise 11
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "country": ["ES", "ES", "FR", "FR", "ES"],
    "age": [25, 40, 35, np.nan, 50]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 3, 3, 5],
    "date": [
        "2026-01-10",
        "2026-02-10",
        "2026-01-15",
        "2026-03-10",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-02-20"
    ],
    "amount": [100, 200, 50, 100, 300, 100, 200, 500]
})

reference_date = pd.Timestamp("2026-04-01")

transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
    last_date=("date", "max"),
    # days_since_last_transaction=("amount", "mean"),
)

transactions["days_since_last_transaction"] = (reference_date - pd.to_datetime(transactions["last_date"])).dt.days
transactions = transactions.drop(columns=["last_date"])

customer_data = customers.merge(transactions, on="customer_id", how="left")

# customer_data["age"] = customer_data["age"].fillna(customer_data["age"].median())

print(customer_data)

   customer_id country   age  total_spend  number_transactions  \
0            1      ES  25.0        300.0                  2.0   
1            2      ES  40.0        150.0                  2.0   
2            3      FR  35.0        600.0                  3.0   
3            4      FR  37.5          NaN                  NaN   
4            5      ES  50.0        500.0                  1.0   

   average_transaction  days_since_last_transaction  
0                150.0                         50.0  
1                 75.0                         22.0  
2                200.0                         31.0  
3                  NaN                          NaN  
4                500.0                         40.0  
   customer_id country   age
0            1      ES  25.0
1            2      ES  40.0
2            3      FR  35.0
3            4      FR   NaN
4            5      ES  50.0


In [82]:
# Exercise 12
df = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "age": [25, 40, 35, 50, 29],
    "monthly_spend": [50, 100, 80, 200, 40],
    "support_calls_next_30d": [2, 5, 0, 8, 1],
    "churn_next_30d": [0, 1, 0, 1, 0]
})

# Knowing that churn_next_30_d is the target we want to predict. We leave that as the target. Related to the features, we
# can not use support_call_next_30d because their values are not present at prediction time so that would be data leakage.
# So, independently of their good predictive power, it can not be used for that reason.

In [84]:
# Exercise 13
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3],
    "date": pd.to_datetime([
        "2026-01-01",
        "2026-02-01",
        "2026-03-15",
        "2026-01-10",
        "2026-03-01",
        "2026-02-15"
    ]),
    "amount": [100, 200, 50, 300, 100, 500]
})

prediction_date = pd.Timestamp("2026-03-01")

customer_spends = transactions[(prediction_date - transactions["date"]).dt.days > 0]

customer_spends = customer_spends.groupby("customer_id").agg(
    total_spend=("amount", "sum")
)

print(customer_spends)

             total_spend
customer_id             
1                    300
2                    300
3                    500


In [87]:
# Exercise 15
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "country": ["ES", "ES", "FR", "FR", "ES"],
    "age": [25, 40, 35, 50, np.nan]
})

transactions = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 3, 3, 3, 5, 5],
    "date": pd.to_datetime([
        "2026-01-10",
        "2026-02-10",
        "2026-01-15",
        "2026-03-10",
        "2026-01-01",
        "2026-02-01",
        "2026-03-01",
        "2026-01-20",
        "2026-02-20"
    ]),
    "amount": [100, 200, 50, 100, 300, 100, 200, 100, 500]
})

reference_date = pd.Timestamp("2026-04-01")

transactions = transactions.groupby("customer_id").agg(
    total_spend=("amount", "sum"),
    number_transactions=("amount", "count"),
    average_transaction=("amount", "mean"),
    last_date=("date", "max"),
)

transactions["days_since_last_transaction"] = (reference_date - transactions["last_date"]).dt.days
transactions.drop(columns=["last_date"])

customer_data = customers.merge(transactions, on="customer_id", how="left")
print(customer_data)

   customer_id country   age  total_spend  number_transactions  \
0            1      ES  25.0        300.0                  2.0   
1            2      ES  40.0        150.0                  2.0   
2            3      FR  35.0        600.0                  3.0   
3            4      FR  50.0          NaN                  NaN   
4            5      ES   NaN        600.0                  2.0   

   average_transaction  last_date  days_since_last_transaction  
0                150.0 2026-02-10                         50.0  
1                 75.0 2026-03-10                         22.0  
2                200.0 2026-03-01                         31.0  
3                  NaN        NaT                          NaN  
4                300.0 2026-02-20                         40.0  
